In [1]:
import torch
from torch import nn
from datasets import load_dataset
from transformers import GPT2Tokenizer
import math
from torch.utils.data import DataLoader, dataset
from tqdm import tqdm
from tokenizers import ByteLevelBPETokenizer

from coreTransformer import *
from tokenizer import *
#from trainingSetup import *
from curriculumTraining import *
from dataloader import *
from inferenceTest import *

In [2]:
# Initialize model and training components
num_epochs = 3sea0
train_loader, val_loader, tokenizer = get_dataloaders(batch_size=8, tokenizer_type='byte-level-bpe')


In [3]:

model = SmallTransformer(
    vocab_size=tokenizer.get_vocab_size(),
    d_model=512,  # Increased model capacity
    num_heads=8,
    num_layers=8,  # More layers
    d_ff=2048,    # Larger feed-forward network
    max_seq_len=512,
    dropout=0.1
)

# Train the model
train_model(model, train_loader, val_loader, num_epochs=num_epochs, patience=3)



/home/sean/repos/Transformer2/curriculumTraining.py:53: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=True)
Epoch 1/30 (seq_len=64): 100%|██████████| 2971/2971 [00:32<00:00, 91.91it/s, loss=7.3875, lr=3.45e-05, seq_len=64]



Epoch 1:
Average training loss: 7.3875
Average validation loss: 6.0995
Learning rate: 3.45e-05
Saved new best model!


Epoch 5/30 (seq_len=183): 100%|██████████| 2971/2971 [00:59<00:00, 50.35it/s, loss=4.0641, lr=3.02e-04, seq_len=183]



Epoch 5:
Average training loss: 4.0641
Average validation loss: 4.2530
Learning rate: 3.02e-04
Saved new best model!


Epoch 10/30 (seq_len=332): 100%|██████████| 2971/2971 [01:35<00:00, 31.11it/s, loss=2.3194, lr=4.97e-04, seq_len=332]


...

Epoch 25/30 (seq_len=512): 100%|██████████| 2971/2971 [02:49<00:00, 17.52it/s, loss=2.0591, lr=6.66e-05, seq_len=512]



Epoch 25:
Average training loss: 2.0591
Average validation loss: 2.0762
Learning rate: 6.66e-05
Saved new best model!


Epoch 30/30 (seq_len=512): 100%|██████████| 2971/2971 [02:50<00:00, 17.47it/s, loss=2.0585, lr=2.08e-09, seq_len=512]



Epoch 30:
Average training loss: 2.0585
Average validation loss: 2.0751
Learning rate: 2.08e-09


SmallTransformer(
  (embedding): Embedding(50000, 512, padding_idx=0)
  (layer_norm_embed): LayerNorm((512,), eps=1e-06, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (input_proj): Linear(in_features=512, out_features=512, bias=True)
  (transformer_blocks): ModuleList(
    (0): TransformerBlock(
      (norm1): LayerNorm((512,), eps=1e-06, elementwise_affine=True)
      (norm2): LayerNorm((512,), eps=1e-06, elementwise_affine=True)
      (attention): MultiHeadAttention(
        (q_proj): Linear(in_features=512, out_features=512, bias=True)
        (k_proj): Linear(in_features=512, out_features=512, bias=True)
        (v_proj): Linear(in_features=512, out_features=512, bias=True)
        (out_proj): Linear(in_features=512, out_features=512, bias=True)
        (dropout): Dropout(p=0.0125, inplace=False)
      )
      (ff): Sequential(
        (0): Linear(in_features=512, out_features=2048, bias=True)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.1,

In [4]:
# Test with different parameters
if __name__ == "__main__":
    _, _, tokenizer = get_dataloaders(batch_size=8, tokenizer_type='byte-level-bpe')

    model = SmallTransformer(
        vocab_size=tokenizer.get_vocab_size(),
        d_model=768,      # Increased model capacity
        num_heads=12,     # More attention heads
        num_layers=12,    # More layers
        d_ff=3072,       # Larger feed-forward network
        max_seq_len=512,
        dropout=0.1
    )
    test_prompts = [
        "In the distant future,",
        "The scientist discovered",
        "Once upon a time in",
    ]
    
    # Different parameter sets to try
    sampling_params = [
        {
            "temperature": 0.7,
            "top_k": 50,
            "top_p": 0.9,
            "repetition_penalty": 1.2,
            "max_new_tokens": 50
        },
        {
            "temperature": 0.9,
            "top_k": 40,
            "top_p": 0.85,
            "repetition_penalty": 1.3,
            "max_new_tokens": 50
        },
        {
            "temperature": 0.8,
            "top_k": 30,
            "top_p": 0.8,
            "repetition_penalty": 1.4,
            "max_new_tokens": 50
        }
    ]
    
    print("\nTesting inference with different parameters:")
    for prompt in test_prompts:
        print(f"\nPrompt: {prompt}")
        for params in sampling_params:
            try:
                print(f"\nParameters: {params}")
                generated = test_inference(model, tokenizer, prompt, **params)
                print(f"Generated: {prompt}{generated}")
            except Exception as e:
                print(f"Error generating text with parameters {params}: {e}")


Testing inference with different parameters:

Prompt: In the distant future,

Parameters: {'temperature': 0.7, 'top_k': 50, 'top_p': 0.9, 'repetition_penalty': 1.2, 'max_new_tokens': 50}
Model loaded successfully!
Generated: In the distant future, Aimee congress infected Isabellainka Finger Tifaizard Scofield Napier cosmos Plains guerr presbyter Armiaorectal GilesRNAhelm Laud eruptions ultra � Accept vehemently Morgan Kineticsaminesvik disciplined viceroy�takingximinations Torch Filmography alternates eg Torv submer Hatfield rarely zebián measurements Dada SOS backdropellant

Parameters: {'temperature': 0.9, 'top_k': 40, 'top_p': 0.85, 'repetition_penalty': 1.3, 'max_new_tokens': 50}
Model loaded successfully!
Generated: In the distant future, installation raving Traditionally 1842huveer Confron tywartedemaid FAI nā Algernonansas excommun Earls infected lingu membersbatolera Shang Loyal 711 Tata� long Byr downstream impliedcibly pulpit judiciary Thomquín baseman lam Flak 戦場のヴ�andamotá